# Fraud Detection With Boosting Workflow

This notebook is a public workflow reconstruction for the fraud-detection project. The original completed notebook was not found, but the local dataset archive was found, so this notebook shows a reproducible boosting pipeline a reviewer can run after placing `creditcard.csv` in `data/raw/`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
DATA_PATH = Path("../data/raw/creditcard.csv")
transactions = pd.read_csv(DATA_PATH)
transactions["Class"].value_counts(normalize=True)

In [ ]:
X = transactions.drop(columns=["Class"])
y = transactions["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

In [ ]:
models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]),
    "decision_tree": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
    "adaboost_tree_stumps": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, class_weight="balanced", random_state=42),
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
    ),
}

In [ ]:
summary = []
for name, model in models.items():
    model.fit(X_train, y_train)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test)[:, 1]
    else:
        prob = model.decision_function(X_test)
    pred = (prob >= 0.5).astype(int)
    summary.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, prob),
        "pr_auc": average_precision_score(y_test, prob),
        "fraud_recall_at_0_50": (confusion_matrix(y_test, pred)[1, 1] / confusion_matrix(y_test, pred)[1].sum()),
    })

pd.DataFrame(summary).sort_values("pr_auc", ascending=False)

In [ ]:
best_model = models["adaboost_tree_stumps"]
prob = best_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, prob)
threshold_table = pd.DataFrame({
    "threshold": np.r_[thresholds, 1.0],
    "precision": precision,
    "recall": recall,
})
threshold_table.sort_values("recall", ascending=False).head(10)

In [ ]:
chosen_threshold = 0.20
pred = (prob >= chosen_threshold).astype(int)
print(classification_report(y_test, pred, target_names=["legitimate", "fraud"]))
print(confusion_matrix(y_test, pred))

## Notes For Reviewers

Fraud datasets are highly imbalanced, so PR-AUC, fraud recall, and false-positive trade-offs are more informative than raw accuracy.